# Objetivo 1 · Segmentación con nnU-Net — prueba en Colab
**Reto CIMAT / BraTS 2018**

Esta es una **prueba**, no el entrenamiento real. Sirve para comprobar que:
1. el dataset convertido (`Dataset501_BraTS2018`) carga y pasa la verificación de nnU-Net;
2. el preprocesamiento y el entrenamiento corren en la GPU de Colab;
3. se pueda medir el **tiempo por época** y la **VRAM usada**, para estimar cuánto tardaría el entrenamiento real.

Entrena **5 épocas del fold 0**. El Dice que sale es bajísimo y no significa nada; solo prueba que el flujo funciona.

### Antes de empezar
1. En tu computadora: `python segmentacion/convertir_nnunet.py --zip` genera `datos/nnunet_raw/Dataset501_BraTS2018.zip` (2.3 GB).
2. Sube ese zip a Google Drive en **`MyDrive/reto_cimat/`**.
3. En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

## 1. GPU disponible

In [ ]:
!nvidia-smi
import os, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno a GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB  | CPUs: {os.cpu_count()}")

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/reto_cimat'
ZIP = f'{DRIVE}/Dataset501_BraTS2018.zip'
assert os.path.exists(ZIP), f'No encuentro {ZIP}: sube el zip a esa carpeta'
print('zip encontrado:', round(os.path.getsize(ZIP) / 1e9, 2), 'GB')

## 3. Instalar nnU-Net v2

In [ ]:
!pip install -q nnunetv2
import importlib.metadata as md
print('nnunetv2', md.version('nnunetv2'))   # anotar esta version en DECISIONES.md

## 4. Carpetas y variables de entorno
- `raw` y `preprocessed` en el disco local de Colab (rápido, pero se pierde al cerrar la sesión).
- `results` en Drive, para que los checkpoints sobrevivan si se corta la sesión.

In [ ]:
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{DRIVE}/nnUNet_results'
# Colab tiene pocos CPUs: el aumento de datos no debe lanzar mas procesos que nucleos
os.environ['nnUNet_n_proc_DA'] = str(os.cpu_count())
# torch.compile agrega minutos de compilacion y a veces falla en T4; para la prueba, apagado
os.environ['nnUNet_compile'] = 'f'
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results'):
    os.makedirs(os.environ[k], exist_ok=True)
    print(k, '=', os.environ[k])

## 5. Copiar y descomprimir el dataset (Drive → disco local)

In [ ]:
%%time
!cp "{ZIP}" /content/
!unzip -q -o /content/Dataset501_BraTS2018.zip -d /content/nnUNet_raw
!rm /content/Dataset501_BraTS2018.zip
RAW = '/content/nnUNet_raw/Dataset501_BraTS2018'
print(len(os.listdir(f'{RAW}/imagesTr')), 'imagenes |', len(os.listdir(f'{RAW}/labelsTr')), 'etiquetas')
# esperado: 1140 imagenes | 285 etiquetas

## 6. Planificación y preprocesamiento
`--verify_dataset_integrity` revisa formas, geometría y etiquetas de los 285 pacientes. `-np 2` limita los procesos para no quedarse sin RAM (Colab gratis tiene unos 12 GB).
Puede tardar entre 30 y 60 minutos.

In [ ]:
%%time
!nnUNetv2_plan_and_preprocess -d 501 -c 3d_fullres --verify_dataset_integrity -np 2

## 7. Usar NUESTROS folds
nnU-Net crearía sus propios folds al azar. Lo reemplazamos por `splits_final.json`, generado desde `particiones/folds.csv`, para que el objetivo 2 use exactamente la misma partición.

In [ ]:
import json, shutil
PRE = '/content/nnUNet_preprocessed/Dataset501_BraTS2018'
shutil.copy(f'{RAW}/splits_final.json', f'{PRE}/splits_final.json')
splits = json.load(open(f'{PRE}/splits_final.json'))
print([(len(s['train']), len(s['val'])) for s in splits])   # esperado: 5 x (228, 57)

plan = json.load(open(f'{PRE}/nnUNetPlans.json'))['configurations']['3d_fullres']
print('patch_size:', plan['patch_size'], '| batch_size:', plan['batch_size'])

## 8. Entrenamiento de prueba: 5 épocas, fold 0
Mientras entrena se registra la VRAM cada 10 s en `/content/vram.log`.
Si `nnUNetTrainer_5epochs` no existe en tu versión, usa `nnUNetTrainer_10epochs`.

In [ ]:
%%time
import subprocess
vram = subprocess.Popen('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 10 > /content/vram.log', shell=True)
!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_5epochs
vram.terminate()

## 9. Resultado de la prueba y estimación del entrenamiento real

In [ ]:
import glob, re, numpy as np
FOLD = f"{os.environ['nnUNet_results']}/Dataset501_BraTS2018/nnUNetTrainer_5epochs__nnUNetPlans__3d_fullres/fold_0"

log = sorted(glob.glob(f'{FOLD}/training_log_*.txt'))[-1]
tiempos = [float(t) for t in re.findall(r'Epoch time: ([\d.]+) s', open(log).read())]
print('tiempo por epoca (s):', tiempos)
t = np.mean(tiempos[1:]) if len(tiempos) > 1 else tiempos[0]   # la primera incluye arranque

usada = max(int(x) for x in open('/content/vram.log').read().split() if x.strip().isdigit())
print(f'VRAM maxima usada: {usada / 1024:.1f} GB de {props.total_memory / 1e9:.1f} GB')

print(f'\nEstimacion con {t:.0f} s/epoca (solo entrenamiento, sin la validacion final):')
for epocas in (100, 250, 1000):
    h = t * epocas / 3600
    print(f'  {epocas:>4} epocas: {h:5.1f} h por fold  | 5 folds: {5 * h:6.1f} h')

resumen = json.load(open(f'{FOLD}/validation/summary.json'))
print('\nDice de validacion (5 epocas: solo comprueba que el flujo funciona):')
for region, nombre in (('(1, 2, 3)', 'WT'), ('(2, 3)', 'TC'), ('(3,)', 'ET')):
    if region in resumen['mean']:
        print(f"  {nombre}: {resumen['mean'][region]['Dice']:.3f}")

## 10. Qué anotar y cómo sigue
Anotar en `DECISIONES.md`: GPU, versión de nnU-Net, `patch_size`/`batch_size`, segundos por época y VRAM máxima.

**Entrenamiento real:** usar `entrenar_colab.ipynb`. Entrena 100 épocas, guarda un checkpoint cada 5, retoma solo si la sesión se corta y guarda el preprocesado en Drive.